# Data Explorer

This notebook, especially its earlier sections, are adapted from a notebook provided in Marks and Tegmark (2023).

In [ ]:
# SECTION: Setup

from visualization_utils import TruthData
from datasets import load_dataset
import torch
from plotly.subplots import make_subplots
import configparser

model = 'llama-2-7b'

# Read settings from config.ini
config = configparser.ConfigParser()
config.read('config.ini')

# Probe layer
layer = 12 # int(config[model]['probe_layer'])

# Whether to strip periods from statements (affects tokenization)
noperiod = config[model].get('noperiod', 'False') == 'True'

# Device for computation
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print(f"Model: {model}")
print(f"Layer: {layer}")
print(f"No period: {noperiod}")
print(f"Device: {device}")

Model: llama-2-7b
Layer: 12
No period: False
Device: cpu


In [ ]:
# SECTION: Basic PCA visualization
TruthData.from_datasets(
    ['facts_true_false'],
    model=model,
    layer=layer,
    center=True,  # Center the data (subtract mean) before PCA
    noperiod=noperiod,  # Whether periods were stripped during generation
    device=device  # CPU or GPU
).plot(
    dimensions=2,  # 2D plot (use 3 for 3D interactive plot)
    dim_offset=1,  # Skip first N principal components (0 = use PC1 and PC2)
    color='label',
)

/Users/peterbennett/Documents/GitHub/geometry-of-truth/acts/llama-2-7b/facts_true_false


In [ ]:
# SECTION: UMAP + PCA visuals

import umap
import plotly.express as px
import pandas as pd
import torch as t
from utils import get_pcs

# Extract activations & metadata
data = TruthData.from_datasets(
    ['facts_true_false'],
    model=model,
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device
)

df = data.df.copy()
acts = t.stack(df['activation'].tolist(), dim=0)

# Run PCA
pcs_tensor = acts.to(device)
pc_proj = t.mm(pcs_tensor, get_pcs(pcs_tensor, 2, offset=1)).cpu().numpy()
df['PC1'] = pc_proj[:, 0]
df['PC2'] = pc_proj[:, 1]

# Run UMAP
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
embedding = reducer.fit_transform(acts.cpu().numpy())
df['UMAP1'] = embedding[:, 0]
df['UMAP2'] = embedding[:, 1]

# Compute metadata features
df['statement_length'] = df['statement'].str.len()
df['token_count']      = df['statement'].str.split().str.len()
df['has_negation']     = df['statement'].str.contains(
    r"\bnot\b|\bno\b|\bnever\b|\bnon\b", case=False, regex=True
).astype(int)
df['has_number']       = df['statement'].str.contains(
    r'\d', regex=True
).astype(int)
df['starts_with_the']  = df['statement'].str.lower().str.startswith('the').astype(int)

# Plotting
features_to_investigate = {
    'label'           : 'Truth label',
    'statement_length': 'Statement length (chars)',
    'token_count'     : 'Token count',
    'has_negation'    : 'Contains negation word',
    'has_number'      : 'Contains a number',
    'starts_with_the' : 'Starts with "The"',
}

for feature, description in features_to_investigate.items():
    shared_kwargs = dict(
        color=feature,
        hover_name='statement',
        hover_data=['label', 'statement_length', 'token_count'],
        color_continuous_scale='inferno',
    )

    fig_pca = px.scatter(df, x='PC1', y='PC2',
                         title=f'PCA — {description}', **shared_kwargs)
    fig_umap = px.scatter(df, x='UMAP1', y='UMAP2',
                          title=f'UMAP — {description}', **shared_kwargs)

    for fig in [fig_pca, fig_umap]:
        fig.update_traces(marker=dict(size=5, opacity=0.7))
        fig.update_yaxes(scaleanchor="x", scaleratio=1)
        fig.update_layout(coloraxis_showscale=True)

    fig_pca.show()
    fig_umap.show()

/Users/peterbennett/Documents/GitHub/geometry-of-truth/acts/llama-2-7b/facts_true_false


/opt/anaconda3/envs/cs229-project/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [ ]:
# SECTION: Separate clusters

from sklearn.mixture import GaussianMixture

# Fit GMM on PCA projection
gmm = GaussianMixture(n_components=2, random_state=42)
df['cluster'] = gmm.fit_predict(df[['PC1', 'PC2']].values)

# Plotting
fig = px.scatter(df, x='PC1', y='PC2',
                 color=df['cluster'].astype(str),
                 hover_name='statement',
                 hover_data=['label', 'token_count'],
                 title='GMM Cluster Assignment')
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

# Print sample statements
for cluster_id in [0, 1]:
    cluster_df = df[df['cluster'] == cluster_id]
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_id} — {len(cluster_df)} statements")
    print(f"Truth balance: {cluster_df['label'].mean():.1%} true")
    print(f"Avg token count: {cluster_df['token_count'].mean():.1f}")
    print(f"{'='*60}")
    for s, label in cluster_df[['statement', 'label']].sample(15).values:
        marker = '✓' if label else '✗'
        print(f"  [{marker}] {s}")


CLUSTER 0 — 520 statements
Truth balance: 48.3% true
Avg token count: 9.0
  [✗] Water is not necessary for human survival.
  [✓] Echoes are sound waves reflecting off surfaces.
  [✗] The fastest mammal in the world is the turtle.
  [✓] The color of the sky is blue.
  [✗] The most populated continent in the world is Antarctica.
  [✗] The three types of clouds are igneous, sedimentary, and metamorphic.
  [✓] Plants use photosynthesis to create energy from sunlight.
  [✗] Sunflowers follow the movement of the moon across the sky.
  [✗] The scientific method is a process for ignoring hypotheses and losing knowledge.
  [✓] The planet Mars is named after the Roman god of war.
  [✗] Giraffes are not real animals, they are made up.
  [✓] The Great Wall of China is the longest wall in the world.
  [✓] Tornadoes are rapidly rotating columns of air.
  [✗] Rainbows form when light avoids water droplets.
  [✗] The human body has 12 pairs of legs.

CLUSTER 1 — 93 statements
Truth balance: 59.1% tru